# HYDRA-Net End-to-End Cascade Benchmark

**Notebook purpose:** Run the full 3-stage cascade on held-out test data, measure real latency distributions, and compare against a monolithic multimodal baseline. Produce publication-ready plots and tables.

**Prerequisites:** You must have already run:
- `01_stage1_dronerf_colab.ipynb` → `stage1_dronerf.json`
- `02_stage2_antiuav_colab.ipynb` → `stage2_antiuav.pt`
- `03_stage3_swarm_colab.ipynb` → `stage3_swarm.pt`

**Runtime:** GPU required for Stages 2/3.


## 1. Setup and load all three stages


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

!git clone https://github.com/YOUR-USERNAME/hydra-net.git
%cd hydra-net
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, 'src')

from hydra_net import HydraCascade
from hydra_net.stage1 import Stage1Triage
from hydra_net.stage2 import CrossModalTransformer, Stage2Module
from hydra_net.stage3 import SwarmReasoningNetwork, Stage3Module

MODELS = '/content/drive/MyDrive/hydra-net-data/models'

stage1 = Stage1Triage.load(f'{MODELS}/stage1_dronerf.json', confidence_threshold=0.95)

s2_model = CrossModalTransformer(embed_dim=192, n_heads=6, n_layers=6, n_classes=2)
s2_model.load_state_dict(torch.load(f'{MODELS}/stage2_antiuav.pt', map_location=device))
stage2 = Stage2Module(s2_model, device=device, confidence_threshold=0.85)

s3_model = SwarmReasoningNetwork(node_dim=128+3+3+3)
s3_model.load_state_dict(torch.load(f'{MODELS}/stage3_swarm.pt', map_location=device))
stage3 = Stage3Module(s3_model, device=device)

cascade = HydraCascade(stage1=stage1, stage2=stage2, stage3=stage3)

## 2. Build test set from multiple scenarios

Mix easy single-drone scenes, hard ambiguous scenes, and swarm scenes to stress-test the cascade's routing behavior.


In [ ]:
import numpy as np

# Load test splits from each stage's dataset preparation step.
# X_test_rf: (N, feature_dim) Stage 1 features from DroneRF test split
# S2_test: list of dicts {'rgb': tensor, 'ir': tensor}
# S3_test: list of dicts {'node_feats': tensor, 'positions': tensor, 'velocities': tensor}

# For demonstration, build random batches here; replace with real loaders.
N = 200
X_test = np.random.randn(N, 30).astype(np.float32)

def random_s2(): return {
    'rgb': torch.randn(3, 224, 224),
    'ir':  torch.randn(1, 224, 224),
}

def random_s3(n_drones=3): return {
    'node_feats': torch.randn(n_drones, 137),
    'positions':  torch.randn(n_drones, 3) * 30,
    'velocities': torch.randn(n_drones, 3) * 5,
}

## 3. Run cascade and measure latency


In [ ]:
import time
from collections import Counter

cascade_lat, monolithic_lat = [], []
exit_stages = []

# Warmup
for _ in range(10):
    cascade.infer(stage1_features=X_test[0], stage2_inputs=random_s2(), stage3_inputs=random_s3())

# Cascade (early-exit enabled)
for i in range(N):
    result = cascade.infer(stage1_features=X_test[i], stage2_inputs=random_s2(), stage3_inputs=random_s3())
    cascade_lat.append(result.total_latency_ms)
    exit_stages.append(result.final_stage)

# Monolithic (force all stages to run)
for i in range(N):
    result = cascade.infer(stage1_features=X_test[i], stage2_inputs=random_s2(), stage3_inputs=random_s3(), force_full=True)
    monolithic_lat.append(result.total_latency_ms)

cascade_lat = np.array(cascade_lat)
monolithic_lat = np.array(monolithic_lat)

print('Exit stage distribution:', Counter(exit_stages))
print()
print(f'{"Metric":<10} {"Cascade":>12} {"Monolithic":>15} {"Speedup":>10}')
for pct in [50, 90, 99]:
    c = np.percentile(cascade_lat, pct)
    m = np.percentile(monolithic_lat, pct)
    print(f'p{pct:<9} {c:>10.2f}ms {m:>13.2f}ms {m/c:>9.1f}x')

cm, mm = cascade_lat.mean(), monolithic_lat.mean()
print(f'{"mean":<10} {cm:>10.2f}ms {mm:>13.2f}ms {mm/cm:>9.1f}x')

## 4. Visualize latency distributions


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(cascade_lat, bins=50, alpha=0.6, label='HYDRA-Net cascade')
axes[0].hist(monolithic_lat, bins=50, alpha=0.6, label='Monolithic baseline')
axes[0].set_xlabel('Latency (ms)'); axes[0].set_ylabel('Count')
axes[0].set_title('Per-sample inference latency'); axes[0].legend()

axes[1].bar(['p50', 'p90', 'p99', 'mean'],
            [np.percentile(cascade_lat, p) if isinstance(p, int) else cascade_lat.mean()
             for p in [50, 90, 99, 'mean']],
            alpha=0.7, label='Cascade')
axes[1].bar(['p50', 'p90', 'p99', 'mean'],
            [np.percentile(monolithic_lat, p) if isinstance(p, int) else monolithic_lat.mean()
             for p in [50, 90, 99, 'mean']],
            alpha=0.7, label='Monolithic')
axes[1].set_ylabel('Latency (ms)'); axes[1].set_title('Percentiles')
axes[1].legend(); axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('results/latency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Final results summary


In [ ]:
import json
from pathlib import Path

results = {
    'n_samples': int(N),
    'device': device,
    'exit_stage_distribution': {str(k): v for k, v in Counter(exit_stages).items()},
    'cascade': {
        'p50_ms': float(np.percentile(cascade_lat, 50)),
        'p90_ms': float(np.percentile(cascade_lat, 90)),
        'p99_ms': float(np.percentile(cascade_lat, 99)),
        'mean_ms': float(cascade_lat.mean()),
    },
    'monolithic': {
        'p50_ms': float(np.percentile(monolithic_lat, 50)),
        'p90_ms': float(np.percentile(monolithic_lat, 90)),
        'p99_ms': float(np.percentile(monolithic_lat, 99)),
        'mean_ms': float(monolithic_lat.mean()),
    },
    'speedup': {
        'p50': float(np.percentile(monolithic_lat, 50) / np.percentile(cascade_lat, 50)),
        'mean': float(monolithic_lat.mean() / cascade_lat.mean()),
    },
}

Path('results').mkdir(exist_ok=True)
with open('results/end_to_end_benchmark.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))